# Exploration des données DVF — Yvelines, 2025

Objectif : comprendre les données avant de construire un dataset de prédiction des prix immobiliers. Cette première version porte sur les Yvelines.

Exécuter les cellules dans l'ordre avec le kernel .venv. Les résultats chiffrés décrits ci-dessous proviennent de l'exploration précédente ; relancer les cellules pour les vérifier.

## 1. Charger les données

`pd.read_csv()` charge le fichier et décompresse automatiquement le format `.gz`. Le chemin relatif suppose un dossier de travail `notebooks/`. `df` conserve les données brutes ; la suite utilisera `df_clean`.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/dvf_78_2025.csv.gz")

## 2. Retirer les colonnes entièrement vides

Premier nettoyage : isna().all() repère les colonnes dont toutes les valeurs sont manquantes. dropna(axis=1, how="all") retire uniquement ces colonnes : axis=1 cible les colonnes et how="all" exige qu'elles soient entièrement vides.

Le résultat est affecté à df_clean ; df reste intact pour les comparaisons. Aucune ligne n'est supprimée. Résultat attendu pour ce fichier : 54 134 lignes et 37 colonnes. Ne pas utiliser how="any", qui supprimerait aussi les colonnes partiellement renseignées.

In [2]:
colonnes_vides = df.columns[df.isna().all()]
print("Colonnes supprimées :", colonnes_vides.tolist())

df_clean = df.dropna(axis=1, how="all")
print("Dimensions après nettoyage :", df_clean.shape)

Colonnes supprimées : ['ancien_code_commune', 'ancien_nom_commune', 'ancien_id_parcelle']
Dimensions après nettoyage : (54134, 37)


## 3. Vérifier les dimensions et afficher un aperçu

`df_clean.shape` donne le nombre de lignes et de colonnes après suppression des colonnes entièrement vides : résultat attendu, 54 134 lignes et 37 colonnes. `head()` affiche les cinq premières lignes. Une ligne ne représente pas nécessairement une transaction distincte.

In [3]:
print("Dimensions :", df_clean.shape)
df_clean.head()

Dimensions : (54134, 37)


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2025-1005318,2025-01-02,1,Vente,15000.0,45.0,NaN,RUE PIERRE CURIE,0550,78370.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1.976941,48.804096
1,2025-1005319,2025-01-03,1,Vente,304000.0,16.0,NaN,RUE ROUGE,2340,78990.0,...,Maison,99.0,5.0,S,sols,NaN,NaN,154.0,1.956081,48.774913
2,2025-1005320,2025-01-02,1,Vente,165300.0,9001.0,NaN,LA CROIX BLANCHE,B015,78390.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.019029,48.799756
3,2025-1005320,2025-01-02,1,Vente,165300.0,8.0,NaN,AV AMBROISE PARE,0025,78390.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.019029,48.799756
4,2025-1005320,2025-01-02,1,Vente,165300.0,8.0,NaN,AV AMBROISE PARE,0025,78390.0,...,Appartement,65.0,3.0,NaN,NaN,NaN,NaN,NaN,2.019029,48.799756


## 4. Examiner les types et les valeurs manquantes

`df_clean.info()` affiche le type et le nombre de valeurs renseignées de chaque colonne restante. Lors de l'exploration initiale, 428 lignes n'avaient pas de `valeur_fonciere`. Les trois colonnes entièrement vides ont déjà été retirées.

`date_mutation` est encore du texte. Les codes géographiques sont des identifiants, même lorsque pandas les lit comme des nombres. Une valeur manquante ne justifie pas à elle seule la suppression d'une ligne.

In [4]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 54134 entries, 0 to 54133
Data columns (total 37 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_mutation                   54134 non-null  str    
 1   date_mutation                 54134 non-null  str    
 2   numero_disposition            54134 non-null  int64  
 3   nature_mutation               54134 non-null  str    
 4   valeur_fonciere               53706 non-null  float64
 5   adresse_numero                45988 non-null  float64
 6   adresse_suffixe               3098 non-null   str    
 7   adresse_nom_voie              53498 non-null  str    
 8   adresse_code_voie             53498 non-null  str    
 9   code_postal                   53498 non-null  float64
 10  code_commune                  54134 non-null  int64  
 11  nom_commune                   54134 non-null  str    
 12  code_departement              54134 non-null  int64  
 13  id_parcelle 

## 5. Identifier les types de locaux

`value_counts()` compte les lignes de chaque catégorie. `dropna=False` inclut les valeurs manquantes dans le résultat.

Le fichier contient 10 670 lignes d'appartements et 8 827 de maisons, soit 19 497 lignes de logements. Ce sont des comptes de lignes, pas encore de logements distincts. Aucun filtre n'est appliqué ici.

In [5]:
df_clean["type_local"].value_counts(dropna=False)

type_local
Dépendance                                  20584
NaN                                         12288
Appartement                                 10670
Maison                                       8827
Local industriel. commercial ou assimilé     1765
Name: count, dtype: int64

## 6. Comprendre les surfaces manquantes

`groupby()` regroupe les lignes par type de local ; `dropna=False` conserve le groupe sans type renseigné. `agg()` calcule deux indicateurs sur la surface : `size` compte toutes les lignes et `count` uniquement les valeurs non manquantes.

Toutes les lignes de maisons et d'appartements ont une surface renseignée. Les surfaces manquent pour toutes les dépendances, toutes les lignes sans type de local et 62 lignes de locaux industriels ou commerciaux. Une surface renseignée n'est pas forcément valide : les zéros et valeurs extrêmes restent à examiner.

In [ ]:
df_clean.groupby("type_local", dropna=False)["surface_reelle_bati"].agg(
    nombre_lignes="size",
    surfaces_renseignees="count"
)

## 7. Distinguer lignes et mutations

`len(df_clean)` compte les lignes et `nunique()` compte les identifiants de mutation distincts. On obtient 54 134 lignes pour 20 924 mutations, soit environ 2,6 lignes par mutation.

L'aperçu initial montre déjà un même identifiant associé à des dépendances et un appartement. Nous devons examiner ces groupes avant de construire les exemples d'entraînement. Supprimer arbitrairement les répétitions d'identifiant pourrait faire perdre des informations.

In [ ]:
nombre_lignes = len(df_clean)
nombre_mutations = df_clean["id_mutation"].nunique()

print("Nombre de lignes :", nombre_lignes)
print("Nombre de mutations distinctes :", nombre_mutations)

## Suite du projet : du dataset aux modèles

1. **Comprendre les mutations sur plusieurs lignes.** Examiner notamment `2025-1005320`, comparer les locaux, parcelles, dispositions et prix. Ne pas sommer les prix répétés ni garder arbitrairement la première ligne.
2. **Définir le périmètre de la première version.** Viser les ventes de maisons et appartements avec un seul logement identifiable par mutation. Examiner les groupes complets avant de filtrer les lignes : plusieurs lignes ne signifient pas nécessairement plusieurs logements. Documenter le traitement des dépendances et exclure les cas ambigus si nécessaire.
3. **Construire une table de modélisation.** Une ligne doit représenter une observation clairement définie. Prévoir la cible `valeur_fonciere`, les caractéristiques disponibles au moment de la prédiction (type, surface, pièces, localisation), ainsi que l'identifiant et la date pour les contrôles et la séparation des données. Vérifier les prix et surfaces manquants, nuls ou incohérents ; justifier chaque exclusion.
4. **Séparer les données avant d'apprendre les transformations.** Vérifier la période réellement couverte, réserver les mutations les plus récentes pour le test et utiliser une validation sur la période antérieure. Une mutation ne doit jamais apparaître dans plusieurs ensembles. Ajuster imputation, encodage et éventuels seuils appris uniquement sur l'entraînement. Ne pas utiliser le prix au m² calculé à partir de la cible comme variable explicative.
5. **Comparer progressivement les modèles.** Commencer par une prédiction constante de la médiane (`DummyRegressor`), puis une régression Ridge et un modèle d'arbres. Regrouper le prétraitement et le modèle dans une pipeline. Comparer la MAE en euros sur la validation, analyser les erreurs par type de bien et gamme de prix, puis évaluer le modèle choisi une seule fois sur le test final.
6. **Rendre le travail reproductible.** Exporter la table dans `data/processed/`, documenter les filtres et les colonnes, puis transférer la préparation et l'entraînement dans des scripts de `src/`. Sauvegarder la pipeline complète avant de passer à FastAPI, Docker et AWS.

La prochaine étape pratique reste l'inspection d'une mutation : le dataset d'entraînement n'est pas encore construit.